# Task 1: Constant Add

Bot task id: `01_constant_add`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/01_constant_add.ipynb)

## Условие

Прибавьте произвольную константу `const_val` к каждому элементу входного вектора `x` и сохраните результат в выходном векторе `z`.

$$z[i] = \text{const\_val} + x[i], \quad i = 0 \dots N-1$$

**Требования**
- одна ось идентификатора программы (pid 0);
- размер блока `B0` равен длине вектора `N` (один блок покрывает весь вектор);
- использовать `tl.arange`, `tl.load` и `tl.store` без маски;
- решение должно работать на GPU (CUDA).

**Подсказка:** индексы можно построить так: `offs = pid * B0 + tl.arange(0, B0)` и затем загрузить `x_ptr + offs`.

`tl.arange` принимает только степень двойки, поэтому проверка берёт `N = 256`.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `01_constant_add.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 01_constant_add.py
import torch
import triton
import triton.language as tl


@triton.jit
def _add_const_kernel(x_ptr, z_ptr, const_val, N: tl.constexpr):
    # YOUR CODE HERE
    pass


def add_triton(x: torch.Tensor, const_val: int) -> torch.Tensor:
    assert x.is_cuda, 'Тензор должен быть на GPU (CUDA).'
    N = x.numel()
    # по условию один блок полностью покрывает вектор
    grid = (1,)
    z = torch.empty_like(x)
    _add_const_kernel[grid](x, z, const_val, N)
    return z

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "01_constant_add.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x = torch.randn(256, device="cuda")
torch.testing.assert_close(sol.add_triton(x, 7), x + 7)

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("01_constant_add.py")